# Testing: Synthetic Data

The `synthetic` module allows us to create artificial datasets with known velocities, extents etc. which we can then compare with those estimated by THUNER.

In [1]:
"""Synthetic data demo/test."""

%load_ext autoreload
%autoreload 2
import xarray as xr
from pathlib import Path
import shutil
import numpy as np
import thuner.data as data
import thuner.default as default
import thuner.track.track as track
import thuner.option as option
import thuner.analyze as analyze
import thuner.data.synthetic as synthetic


Welcome to the Thunderstorm Event Reconnaissance (THUNER) package 
v0.0.16! This package is still in testing and development. Please 
visit github.com/THUNER-project/THUNER for examples, and to report 
issues or contribute.
 
THUNER is a flexible toolkit for performing multi-feature detection, 
tracking, tagging and analysis of events within meteorological datasets. 
The intended application is to convective weather events. For examples 
and instructions, see https://github.com/THUNER-project/THUNER and 
https://thuner.readthedocs.io/en/latest/. If you use THUNER in your 
research, consider citing the following papers;

Short et al. (2023), doi: 10.1175/MWR-D-22-0146.1
Raut et al. (2021), doi: 10.1175/JAMC-D-20-0119.1
Fridlind et al. (2019), doi: 10.5194/amt-12-2979-2019
Whitehall et al. (2015), doi: 10.1007/s12145-014-0181-3
Dixon and Wiener (1993), doi: 10.1175/1520-0426(1993)010<0785:TTITAA>2.0.CO;2
Leese et al. (1971), doi: 10.1175/1520-0450(1971)010<0118:AATFOC>2.0.CO;2



## Geographic Coordinates

In [2]:
# Set a flag for whether or not to remove existing output directories
remove_existing_outputs = True

# Parent directory for saving outputs
base_local = Path.home() / "THUNER_output"
start = "2005-11-13T00:00:00"
end = "2005-11-13T01:00:00"

output_parent = base_local / "runs/synthetic/geographic"

In [3]:
if output_parent.exists() and remove_existing_outputs:
    shutil.rmtree(output_parent)

In [4]:
options_directory = output_parent / "options"
options_directory.mkdir(parents=True, exist_ok=True)

# Create a grid
lat = np.arange(-14, -6 + 0.025, 0.025).tolist()
lon = np.arange(128, 136 + 0.025, 0.025).tolist()
grid_options = option.grid.GridOptions(name="geographic", latitude=lat, longitude=lon)
grid_options.to_json(options_directory / "grid.json")

# Initialize synthetic objects
starting_objects = []
for i in range(5):
    major = 2 * (7 + 4 * i)  # full axis length in km
    obj = synthetic.EllipsoidObject(
        time=start,
        center_latitude=np.mean(lat),
        center_longitude=lon[(i + 1) * len(lon) // 6],
        direction=-np.pi / 4 + i * np.pi / 8,
        speed=30 - 4 * i,
        major=major,
        minor=0.4 * major,
        orientation=0.25 * np.pi + i * np.pi / 8,
    )
    starting_objects.append(obj)
# Create data options dictionary
synthetic_options = data.synthetic.SyntheticOptions(objects=starting_objects)
data_options = option.data.DataOptions(datasets=[synthetic_options])
data_options.to_json(options_directory / "data.json")

track_options = default.track.synthetic_track()
track_options.to_json(options_directory / "track.json")

# Create the display_options dictionary
visualize_options = default.visualize.synthetic_runtime(
    options_directory / "visualize.json"
)
visualize_options.to_json(options_directory / "visualize.json")

2026-06-04 20:24:35,606 - thuner.option.grid - WARNING - altitude not specified. Using default altitudes.
2026-06-04 20:24:35,607 - thuner.option.grid - WARNING - shape not specified. Will attempt to infer from input.


In [5]:
visualize_options.model_dump()

{'type': 'RuntimeOptions',
 'objects': {'convective': {'type': 'ObjectRuntimeOptions',
   'parent_local': PosixPath('/home/ewan/THUNER_output/runs/synthetic/geographic/options/visualize.json'),
   'style': 'presentation',
   'weights_filepath': None,
   'name': 'convective',
   'figures': [{'type': 'FigureOptions',
     'name': 'match',
     'function': 'thuner.visualize.runtime.visualize_tint_match',
     'style': 'presentation',
     'animate': True,
     'single_color': False,
     'template': None}],
   'animate': True,
   'single_color': False}}}

In [6]:
times = np.arange(
    np.datetime64(start),
    np.datetime64(end) + np.timedelta64(10, "m"),
    np.timedelta64(10, "m"),
)
track.track(
    times=times,
    data_options=data_options,
    grid_options=grid_options,
    track_options=track_options,
    visualize_options=visualize_options,
    output_directory=output_parent,
)

2026-06-04 20:24:44,890 - thuner.track.track - INFO - Beginning thuner tracking. Saving output to /home/ewan/THUNER_output/runs/synthetic/geographic.
2026-06-04 20:24:44,892 - thuner.track.track - INFO - Processing 2005-11-13T00:00:00.
2026-06-04 20:24:44,893 - thuner.data.synthetic.options - INFO - Updating synthetic dataset for 2005-11-13T00:00:00.


2026-06-04 20:24:46,400 - thuner.track.track - INFO - Processing hierarchy level 0.
2026-06-04 20:24:46,401 - thuner.track.track - INFO - Tracking convective.
2026-06-04 20:24:46,404 - thuner.utils - INFO - Compiling thuner.detect.steiner.steiner_scheme with Numba. Please wait.
2026-06-04 20:24:46,533 - thuner.match.match - INFO - Matching convective objects.
2026-06-04 20:24:46,534 - thuner.match.match - INFO - No current mask, or no objects in current mask.
2026-06-04 20:24:46,537 - thuner.visualize.runtime - INFO - Creating runtime visualization figures.
2026-06-04 20:24:53,635 - thuner.track.track - INFO - Processing 2005-11-13T00:10:00.
2026-06-04 20:24:53,636 - thuner.data.synthetic.options - INFO - Updating synthetic dataset for 2005-11-13T00:10:00.
2026-06-04 20:24:54,933 - thuner.track.track - INFO - Processing hierarchy level 0.
2026-06-04 20:24:54,934 - thuner.track.track - INFO - Tracking convective.
2026-06-04 20:24:54,937 - thuner.write.mask - INFO - Writing convective ma

![THUNER applied to synthetic data.](https://raw.githubusercontent.com/THUNER-project/THUNER/refs/heads/main/gallery/synthetic.gif)

## Cartesian Coordinates

In [6]:
central_latitude = -10
central_longitude = 132

y = np.arange(-400e3, 400e3 + 2.5e3, 2.5e3).tolist()
x = np.arange(-400e3, 400e3 + 2.5e3, 2.5e3).tolist()

grid_options = option.grid.GridOptions(
    name="cartesian",
    x=x,
    y=y,
    central_latitude=central_latitude,
    central_longitude=central_longitude,
)
grid_options.to_json(options_directory / "grid.json")

2026-06-03 23:37:54,422 - thuner.option.grid - WARNING - altitude not specified. Using default altitudes.


In [ ]:
output_parent = base_local / "runs/synthetic/cartesian"
if output_parent.exists() & remove_existing_outputs:
    shutil.rmtree(output_parent)
    
times = np.arange(
    np.datetime64(start),
    np.datetime64(end) + np.timedelta64(10, "m"),
    +np.timedelta64(10, "m"),
)

track.track(
    times=times,
    data_options=data_options,
    grid_options=grid_options,
    track_options=track_options,
    visualize_options=None,
    output_directory=output_parent,
)

2026-06-03 23:37:55,143 - thuner.track.track - INFO - Beginning thuner tracking. Saving output to /home/ewan/THUNER_output/runs/synthetic/cartesian.
2026-06-03 23:37:55,145 - thuner.track.track - INFO - Processing 2005-11-13T00:00:00.
2026-06-03 23:37:55,146 - thuner.data.synthetic.options - INFO - Updating synthetic dataset for 2005-11-13T00:00:00.


2026-06-03 23:37:56,331 - thuner.track.track - INFO - Processing hierarchy level 0.
2026-06-03 23:37:56,331 - thuner.track.track - INFO - Tracking convective.
2026-06-03 23:37:56,360 - thuner.match.match - INFO - Matching convective objects.
2026-06-03 23:37:56,360 - thuner.match.match - INFO - No current mask, or no objects in current mask.
2026-06-03 23:37:56,363 - thuner.visualize.runtime - INFO - Creating runtime visualization figures.
2026-06-03 23:37:58,238 - thuner.track.track - INFO - Processing 2005-11-13T00:10:00.
2026-06-03 23:37:58,239 - thuner.data.synthetic.options - INFO - Updating synthetic dataset for 2005-11-13T00:10:00.
2026-06-03 23:37:59,290 - thuner.track.track - INFO - Processing hierarchy level 0.
2026-06-03 23:37:59,290 - thuner.track.track - INFO - Tracking convective.
2026-06-03 23:37:59,293 - thuner.write.mask - INFO - Writing convective masks to /home/ewan/THUNER_output/runs/synthetic/cartesian/output.zarr::masks/convective.
2026-06-03 23:37:59,327 - thuner

In [11]:
ground_truth = analyze.synthetic.write_ground_truth(
    output_parent, data_options=data_options, times=times
)
print(ground_truth["synthetic"].head(10).to_string())

2026-06-04 20:51:03,574 - thuner.analyze.synthetic - INFO - Wrote ground truth for synthetic.


                        latitude  longitude     u     v  major  minor  orientation  eccentricity  intensity
time                id                                                                                     
2005-11-13 00:00:00 0   -10.0000   129.3250 -21.2  21.2   14.0    5.6       0.7854        0.9165      69.25
                    1   -10.0000   130.6750  -9.9  24.0   22.0    8.8       1.1781        0.9165      69.25
                    2   -10.0000   132.0250   0.0  22.0   30.0   12.0       1.5708        0.9165      69.25
                    3   -10.0000   133.3500   6.9  16.6   38.0   15.2       1.9635        0.9165      69.25
                    4   -10.0000   134.7000   9.9   9.9   46.0   18.4       2.3562        0.9165      69.25
2005-11-13 00:10:00 0    -9.8849   129.2090 -21.2  21.2   14.0    5.6       0.7854        0.9165      69.25
                    1    -9.8697   130.6206  -9.9  24.0   22.0    8.8       1.1781        0.9165      69.25
                    2    -9.